# Regression with a Deep Neural Network (DNN) using PyTorch

Here's a __deep feedforward neural network__ using __two hidden layers__ and an output layer to sum up two input values.
This notebook highlights: 
  * Eager Execution mode
  * Use of the computational graph
  * Use of Autograd (the automatic differentiation engine)
  * How to build a DNN

Let's first import the necessary libraries.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## How PyTorch builds a computational graph: eager execution and Autograd

PyTorch normally runs in **Eager Execution** mode: Python statements and tensor
operations execute immediately, one after another. During the forward pass,
PyTorch also records the operations that involve tensors requiring gradients.
Those recorded operations form a **computational graph** (more precisely, a
dynamic graph / autograd graph).

- A tensor created with `requires_grad=True` asks Autograd to track operations
  involving that tensor.
- Each differentiable operation creates a result tensor with a `grad_fn`
  (unless the result is a leaf tensor). The `grad_fn` links the result to the
  operation that produced it.
- The graph is built as the forward computation runs. It can therefore reflect
  the operations actually executed in that pass, including Python control flow.
- Calling `backward()` traverses the graph in reverse and computes gradients
  using the chain rule. For a leaf tensor, the gradient is available in `.grad`.
- The graph is normally freed after `backward()` to release memory. A new
  forward pass builds a new graph; this is why PyTorch's graph is called dynamic.

The next cell is a small, standalone demonstration. It is separate from the
neural network training below.

In [ ]:
# Eager execution: each statement executes immediately.
# requires_grad=True tells Autograd to track operations on x.
x = torch.tensor(2.0, requires_grad=True)
print("x:", x)
print("x.requires_grad:", x.requires_grad)
print("x is a leaf tensor:", x.is_leaf)

# These operations run now and are recorded by Autograd.
a = x * 3
b = a + 4
y = b ** 2

print("\\nIntermediate/result tensors:")
for name, tensor in [("a = x * 3", a), ("b = a + 4", b), ("y = b ** 2", y)]:
    print(f"{name}: value={tensor.item()}, requires_grad={tensor.requires_grad}, grad_fn={tensor.grad_fn}")

# Backpropagation: dy/dx = 2 * (3*x + 4) * 3.
y.backward()
print("\\nAutograd result:")
print("x.grad =", x.grad)
print("Expected gradient =", 2 * (3 * x.item() + 4) * 3)

The DeepFeedForwardNN class defines a deep feedforward neural network using PyTorch. The neural network consists of an input layer, two hidden layers with ReLU activations, and an output layer. The input layer has two nodes, both hidden layers have ten nodes, and the output layer has one node. The __init__() method initialises the network architecture based on the specified input, hidden, and output dimensions, while the forward() method defines the flow of data through these layers to generate the output predictions.

In [ ]:
class DeepFeedForwardNN(nn.Module):
    
    """
    Class initialisation method defines deep feedforward neural network architecture with two hidden layers
    """
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, output_dim):
        super(DeepFeedForwardNN, self).__init__()
        # Add a fully connected layer from input to first hidden layer
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        # Use ReLU non-linear activation function after first hidden layer  
        self.relu1 = nn.ReLU()
        # Fully connected layer from first hidden layer to second hidden layer
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        # Use ReLU non-linear activation function after second hidden layer
        self.relu2 = nn.ReLU()
        # Fully connected layer from second hidden layer to output 
        self.fc3 = nn.Linear(hidden_dim2, output_dim)  

    """ defines the forward pass of the neural network """
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x


## Inspect the computational graph for this neural network

A `nn.Linear` layer has trainable parameters (weights and biases). These
parameters have `requires_grad=True` by default, so Autograd tracks how the
loss depends on them. The input data does not need `requires_grad=True` when
the goal is to calculate gradients for model parameters.

In the next cell, we run one forward pass and calculate a loss. Because this
is Eager Execution, the graph is created as these Python operations execute.
Inspect `grad_fn` to see that the predictions and loss are connected to
Autograd. Then `loss.backward()` asks Autograd to calculate parameter
gradients. The training loop below repeats this forward/build-graph,
backward, and optimiser-update sequence for every epoch.

In [ ]:
# Use a separate demonstration model so this inspection does not alter
# the model that will be trained in main().
demo_model = DeepFeedForwardNN(input_dim=2, hidden_dim1=10, hidden_dim2=10, output_dim=1)

print("Do model parameters require gradients?")
for name, parameter in demo_model.named_parameters():
    print(f"{name}: requires_grad={parameter.requires_grad}, is_leaf={parameter.is_leaf}")

# A normal input tensor is sufficient: gradients are needed for the parameters.
demo_input = torch.tensor([[1.0, 2.0]])
demo_target = torch.tensor([[3.0]])

# Forward pass: each layer operation executes immediately and contributes
# to the dynamic autograd graph.
demo_prediction = demo_model(demo_input)
demo_loss = nn.MSELoss()(demo_prediction, demo_target)

print("\\nGraph-related information:")
print("Input requires_grad:", demo_input.requires_grad)
print("Prediction requires_grad:", demo_prediction.requires_grad)
print("Prediction grad_fn:", demo_prediction.grad_fn)
print("Loss requires_grad:", demo_loss.requires_grad)
print("Loss grad_fn:", demo_loss.grad_fn)

# Backward pass: Autograd traverses the graph and fills parameter .grad fields.
demo_loss.backward()
print("\\nExample parameter gradients after backward():")
for name, parameter in demo_model.named_parameters():
    print(f"{name}: gradient available={parameter.grad is not None}, "
          f"gradient shape={None if parameter.grad is None else tuple(parameter.grad.shape)}")

The train_model() function trains the neural network over 1,000 epochs. In each epoch, the model generates predictions, calculates the loss, computes gradients through backpropagation, and updates the model parameters using the optimiser. Training progress is displayed every 100 epochs to monitor the reduction in loss. Then Epochs vs Loss is plotted.

In [ ]:
def train_model(sample_data, labels, model, loss_fn, optimizer):
    """
    Training loop for the neural network
    """
    num_epochs = 1000
    epoch_losses = []

    for epoch in range(num_epochs):
        # EAGER FORWARD PASS: these operations execute immediately. PyTorch
        # builds a fresh dynamic autograd graph for this epoch.
        predictions = model(sample_data)
        # The loss is also part of the graph, connecting predictions to parameters.
        loss = loss_fn(predictions, labels)
        # Clear gradients from the previous optimisation step (gradients accumulate by default).
        optimizer.zero_grad()
        # AUTOGRAD: traverse the graph backwards and compute d(loss)/d(parameter).
        loss.backward()
        # Optimiser uses the gradients computed by Autograd to update parameters.
        optimizer.step()

    # Store the loss for plotting
        epoch_losses.append(loss.item())
    
    # Print progress
        if (epoch+1) % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.6f}')

    # Plot Epoch vs Loss
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, num_epochs + 1), epoch_losses, label='Training loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Epoch vs Loss')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return epoch_losses


The main() function configures the deep neural network, defines the training data and target labels (the sum of the two input values), and initialises the model. It uses mean squared error (MSE) as the loss function and stochastic gradient descent (SGD) as the optimiser to train the model. Finally, the trained network is evaluated on unseen test data to generate predictions, with gradient computation disabled during testing.

During training, `loss.backward()` invokes the Autograd engine. The optimiser
then updates the parameters using the gradients stored in each parameter's
`.grad` attribute. During testing, `torch.no_grad()` disables gradient tracking:
no backward pass is required, so building an autograd graph would be unnecessary.


In [ ]:
def main():
    """
    Sets up NN configuration and training data, then trains the model and tests it with new data
    """
    # Sample data for training the neural network
    input_dim = 2   # Two input values
    hidden_dim1 = 10 # Ten neurons in the first hidden layer
    hidden_dim2 = 10 # Ten neurons in the second hidden layer
    output_dim = 1  # One output value (the sum of two input values)
    sample_data = torch.tensor([[1.0, 2.0], [2.0, 3.0], [3.0, 4.0], [4.0, 5.0], [5.0, 6.0]])
    print(f"Shape of sample_data = {sample_data.shape}")
    print(f"sample_data = \n {sample_data}")

    # Define the ground truth labels (the sum of two input values)
    # Sum the input values in each row (i.e., dim=1)
    labels = torch.sum(sample_data, dim=1).unsqueeze(1)  # Unsqueeze to match the shape of predictions
    print(f"Shape of labels = {labels.shape}")
    print(f"labels = \n {labels}")

    # Create an instance of the deep neural network
    model = DeepFeedForwardNN(input_dim, hidden_dim1, hidden_dim2, output_dim)

    # Use the Mean Square Error loss function
    loss_fn = nn.MSELoss()

    # Define the optimizer
    optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)
    train_model(sample_data, labels, model, loss_fn, optimizer)
    
    # Test the model with new *unseen* data
    test_data = torch.tensor([[6.0, 7.0], [7.0, 8.0], [8.0, 9.0]])
    with torch.no_grad():
        test_predictions = model(test_data)

    print("\nTest data:")
    print(test_data)
    print("\nPredictions for test data:")
    print(test_predictions)

if __name__ == "__main__":
    main()


Play with the __training hyperparamter__ learning rate, *lr*, above and see how the "Epoch vs Loss' plot changes; for example, try from 0.1 to 0.0001, decreasing by tength each time. What is happening to the accuracy of the test data? What would be an acceptable learning rate __without__unneccessarily increasing computation and delaying the training?